In [5]:
import os
from pathlib import Path
from dotenv import load_dotenv

def load_project_env() -> Path:
    """Load the first .env found from the current folder up to the workspace root."""
    for candidate_dir in [Path.cwd(), *Path.cwd().resolve().parents]:
        candidate = candidate_dir / ".env"
        if candidate.exists():
            load_dotenv(candidate, override=True)
            return candidate
    raise FileNotFoundError("No se encontró el archivo .env")

env_file = load_project_env()
print(f"Loaded environment from: {env_file}")

# Variables cruciales para LangSmith

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = str(os.getenv("POSTGRESQL_MCP_LANGSMITH") or None)
os.environ["LANGSMITH_PROJECT"] = "Postresql MCP"

os.environ["NVIDIA_API_KEY"] = str(os.getenv("NVIDIA_API_KEY") or None)
os.environ["ORCHESTRATOR_API_KEY_LOCAL"] = str(os.getenv("ORCHESTRATOR_API_KEY_LOCAL") or None)
os.environ["ORCHESTRATOR_BASE_URL_LOCAL"] = str(os.getenv("ORCHESTRATOR_BASE_URL_LOCAL") or None)


from typing import Annotated, TypedDict
from langchain_mcp_adapters.tools import load_mcp_tools
from langgraph.graph.message import add_messages
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.checkpoint.memory import MemorySaver
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

Loaded environment from: /home/santi/Documentos/LangGraph/.env


In [6]:
system_env = dict(os.environ)

pg_user = os.getenv("PG_USER")
pg_password = os.getenv("PG_PASSWORD")
pg_host = os.getenv("PG_HOST")
pg_port = os.getenv("PG_PORT") or "5433"
pg_database = os.getenv("PG_DATABASE")

# MCP Client Configuration
client = MultiServerMCPClient(
    {
        "postgresql": {
            "command": "npx",
            "args": [
                "-y",
                "@modelcontextprotocol/server-postgres",
                f"postgresql://{pg_user}:{pg_password}@{pg_host}:{pg_port}/{pg_database}",
            ],
            "transport": "stdio",
            "env": system_env,
        }
    }
)

In [7]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    query: str
    is_safe: bool
    error: str

In [8]:
llm = ChatOpenAI(
    model="z-ai/glm-5.1",
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1", # NVIDIA's API URL
    temperature=0.0,
)

In [12]:
from langgraph.prebuilt import tools_condition
from langchain_core.tracers.context import tracing_v2_enabled


def handle_tool_error(error: Exception) -> str:
    return (
        "The database tool failed while executing a query. "
        f"Error detail: {error}. "
        "Do not assume table or column names. Reinspect the schema or metadata, "
        "then retry with a validated query."
    )

def should_call_tools(state: State) -> str:
    is_safe = state["is_safe"]
    error = state["error"]

    if not is_safe:
        print(f"Guardrail detected an unsafe query. Error: {error}")
        return "end"
    
    return "tools"  # Query is safe, proceed without calling tools
    

async def run_agent():
    async with client.session("postgresql") as mcp_session:
        tools = await load_mcp_tools(mcp_session)
        agent = llm.bind_tools(tools)

        # Call de Agent
        def call_sql_agent(state: State):
            # Get messages from state
            messages = state["messages"]

            # Define Prompt
            agent_prompt = ("system", 
                "You are a expert assistant connected to the SEDICI database through MCP."
                "STRICT RULES: \n"
                "1. NEVER guess or asume the name of the tables, columns or any other database structure."
                "2. Your first step ALWAYS has to be use the tools to explore the database schemas available and get the information needed to answer the user's question (views) ."
                "3. Once you have the tables that seems relevant, inspect the structure (views, types of the columns, etc) before trying to answer the user's question."
                "4. If the query fails, read the error message, understand what went wrong, and use the tools again to get the correct information or fix the query before trying to answer the user's question again."

                "Answer the question based on the provided messages and using the tools when necessary."
            )   

            final_prompt = [agent_prompt] + messages
            # Answer
            response = agent.invoke(final_prompt)

            query = ""
            if response.tool_calls:
                for tc in response.tool_calls:
                    if "query" in tc["args"]:
                        query = tc["args"]["query"]
                        break

            query = "DROP TABLE users;"  # Simulating an unsafe query for testing purposes

            return {
                "messages": response,
                "query": query,
            }
        
        def call_guardrail(state: State):
            query = state["query"]

            unsafe_keywords = ["DROP", "DELETE", "ALTER", "UPDATE", "INSERT"]
            if any(keyword in query.upper() for keyword in unsafe_keywords):
                return {"is_safe": False, "error": "Query contains potentially harmful operations."}
            
            return {"is_safe": True, "error": ""}

        # Nodes
        workflow = StateGraph(State)
        workflow.add_node("agent", call_sql_agent)
        workflow.add_node("guardrail", call_guardrail)
        workflow.add_node("tools_node", ToolNode(tools, handle_tool_errors=handle_tool_error))

        # Edges
        workflow.add_edge(START, "agent")
        workflow.add_edge("agent", "guardrail")

        workflow.add_conditional_edges(
            "guardrail", should_call_tools,
            {
                "tools": "tools_node",
                "end": END,
            },
        )
        workflow.add_edge("tools_node", "agent")

        # Compile graph
        app = workflow.compile(checkpointer=MemorySaver())

        # Execution
        inputs = {
            "messages": [
                ("user", "How many authors are registered in the system?"),
            ]
        }
        config = {"configurable": {"thread_id": "1"}}

        with tracing_v2_enabled():
            result = await app.ainvoke(inputs, config=config)
    
        # print(result["messages"][-1]) 

await run_agent()

Guardrail detected an unsafe query. Error: Query contains potentially harmful operations.
